# DINOv3 License-Aware Embedding Tutorial

This tutorial compares DINOv2 and DINOv3 embedding pipelines, explains license
constraints, and demonstrates the BYOT path for accessing DINOv3.

**DINOv2:** Apache-2.0 — open access  
**DINOv3:** Research-only license — gated HF repo (`auth_required`)

## DINOv2 vs DINOv3 Comparison

| Feature | DINOv2 | DINOv3 |
|---|---|---|
| License | Apache-2.0 | Research-only |
| HF access | Open | Gated |
| Architecture | ViT-S/B/L/g | ViT-B/L/g+ |
| Training data | LVD-142M | LVD-142M + curated |
| Linear probe ImageNet | 79.2 (ViT-B) | 82.1 (ViT-B) |
| Self-supervised features | yes | yes (improved) |
| VisionServeX status | ready | auth_required |

In [ ]:
# Cell 3: Run DINOv2-base (Apache-2.0, works without auth)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'visionservex', '-q'])

import visionservex as vsx
import numpy as np
from visionservex.dino import run_dinov2_embed

# Synthetic 224x224 image batch
images = np.random.randint(0, 255, (4, 224, 224, 3), dtype=np.uint8)

result = run_dinov2_embed(model_id='dinov2-base', images=images)
print(f'DINOv2-base status     : {result["status"]}')
print(f'Embedding shape        : {result["embedding_shape"]}')
print(f'Embedding dim          : {result["embedding_dim"]}')
print(f'Latency ms             : {result["latency_ms"]}')

In [ ]:
# Cell 4: kNN self-similarity on DINOv2 embeddings
from visionservex.dino import compute_knn_similarity
import matplotlib.pyplot as plt

sim_result = compute_knn_similarity(embeddings=result['embeddings'], k=3)
print(f'kNN similarity matrix shape: {sim_result["similarity_matrix_shape"]}')
print(f'Top-1 neighbours: {sim_result["top1_indices"]}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(sim_result['similarity_matrix'], cmap='hot', vmin=0, vmax=1)
ax.set_title('DINOv2-base: kNN self-similarity (4 images)')
ax.set_xlabel('Image index'); ax.set_ylabel('Image index')
plt.colorbar(ax.images[0], ax=ax)
plt.tight_layout()
plt.savefig('/tmp/dinov2_knn_sim.png', dpi=80)
plt.show()

In [ ]:
# Cell 5: Check DINOv3 status — expected: auth_required
from visionservex.dino import get_dinov3_status

v3_status = get_dinov3_status()
print(f'DINOv3 status        : {v3_status["status"]}')
print(f'HF repo              : {v3_status.get("hf_repo", "N/A")}')
print(f'Gated                : {v3_status.get("gated", False)}')
print(f'License              : {v3_status.get("license", "N/A")}')
print(f'Token present        : {v3_status.get("token_present", False)}')
if v3_status.get('message'):
    print(f'Message              : {v3_status["message"]}')

## DINOv3 License Explanation

DINOv3 is released under Meta AI's **Research License**, which restricts:
- Commercial use (not permitted without separate agreement)
- Redistribution of model weights
- Use in production systems without Meta AI approval

This differs from DINOv2 (Apache-2.0), which allows commercial use.

**VisionServeX policy:** gated models (DINOv3, SAM3) require the user to provide
their own HuggingFace token after accepting the model license on HuggingFace Hub.
VisionServeX never caches or redistributes gated weights.

In [ ]:
# Cell 7: BYOT path for DINOv3
from visionservex.dino import get_dinov3_byot_instructions

byot = get_dinov3_byot_instructions()
print(f'BYOT status   : {byot["status"]}')
print(f'HF page       : {byot.get("hf_page", "N/A")}')
print()
print('Steps to unlock DINOv3:')
for i, step in enumerate(byot.get('steps', []), 1):
    print(f'  {i}. {step}')

## Next Steps

- For GroundingDINO text-prompted detection, see `06_grounding_dino_text_detect.ipynb`.
- For SAM3 BYOT workflow, see `04_sam3_byot_auth.ipynb`.
- DINOv2 large model: replace `dinov2-base` with `dinov2-large` in Cell 3.